# Preclass
AI for Materials Science — Hands-on session 5: Train and Test ML/DL Models

This is a warm-up for the two tools you will use in class: scikit-learn for classical machine
learning and PyTorch for a small neural network. It takes about 40 minutes. There is nothing to submit.

Data: the Perch rows of fish.csv from the hg-mldl repository of Haesun Park's machine-learning textbook (MIT License); original measurements from Lake Laengelmavesi, Finland (J. Puranen, Journal of Statistics Education data archive). In class we move on to battery electrode data from the Materials Project.

## What this notebook covers

### 0 · Setup
Check the libraries and download the course material.

### A · scikit-learn: fit, predict, score
Load a small table of fish, split it, fit a linear model, and read three error metrics. Then let the line bend with polynomial features.

### B · PyTorch: tensors, gradients, a training loop
See what a tensor is, let PyTorch compute a gradient, and train a one-neuron model in a ten-line loop. Finally, see why one neuron cannot learn XOR and one hidden layer can.

---

**First:** File ▸ Save a copy in Drive, and work in your copy.

Run the cells one at a time from the top. Later cells reuse variables created in earlier ones, so skipping ahead gives you a "name is not defined" error. Lines starting with `##` inside the code are comments written for you; Python does not run them.

**Shared or lab PC?** Sign out of Google when you finish and choose Runtime ▸ Disconnect and delete runtime.

## 0. Setup

### 0-1. Check the libraries
Everything this notebook needs ships with Colab: NumPy, pandas, scikit-learn, Matplotlib and PyTorch.
There is nothing to install; this cell only prints the versions you are running.

In [ ]:
import numpy, pandas, sklearn, matplotlib, torch
modules = [("numpy", numpy), ("pandas", pandas), ("scikit-learn", sklearn),
           ("matplotlib", matplotlib), ("torch", torch)]
for name, module in modules:
    print(f"{name:13s} {module.__version__}")

### 0-2. Download the course material
The same repository as in the earlier sessions. Its `Data/` folder holds the fish table for this
notebook and the electrode data for class. If you run the cell a second time, git answers that the
destination path already exists: the folder is already there, and you can carry on.

In [ ]:
## A leading ! runs a terminal command instead of Python.
!git clone https://github.com/kwongibaek/MS49900-AI4M.git

### 0-3. Import the libraries
`import ... as` gives a library a short name; `np`, `pd` and `plt` are the usual ones. From scikit-learn we
take only the pieces we use: the linear model, polynomial features with a scaler and a pipeline (A-6), three error metrics and the function that splits the data.
`torch.nn` holds the building blocks of neural networks.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
import torch
import torch.nn as nn

### 0-4. Set the file paths
We name the data folder once and call it `DATA_DIR` from here on. If the cell prints `False`, the download
in 0-2 did not finish: run 0-2 again.

In [ ]:
DATA_DIR = Path("MS49900-AI4M/Data")      ## course data
OUTPUT_DIR = Path("outputs")             ## figures you make today
OUTPUT_DIR.mkdir(exist_ok=True)
print("Data folder exists:", DATA_DIR.exists())

## A. scikit-learn: fit, predict, score

Every supervised model in this course follows the same four steps: **split** the data, **fit** a
model on the training part, **predict** on the test part, and **score** the predictions. We practise
the steps on a small real dataset: perch, a freshwater fish, measured by length, height and width.

### A-1. Load the perch data
`fish.csv` lists 159 fish of seven species caught in one Finnish lake. We keep the 56 perch. Weight
is in grams; length, height and width are in centimetres.

`fish["Species"] == "Perch"` gives one True/False value per row, and putting it in square brackets keeps
only the True rows: the same mask idea as in session 1. `head()` shows the first five rows. `Diagonal` is
another length measurement that we leave out.

In [ ]:
fish = pd.read_csv(DATA_DIR / "fish.csv")
perch = fish[fish["Species"] == "Perch"]   ## keep one species
print("fish:", len(fish), " perch:", len(perch))
perch.head()

### A-2. Split into training and test sets
The model never sees the test rows while it is being fitted. That is what makes the test score an
honest estimate of how it will do on new data.

By convention `X` (capital) is the feature table, one column per input, and `y` is the target, one value
per row. `test_size=0.2` sends 20% of the fish to the test set. `random_state=0` fixes the shuffle, so you
get the same split every time you run the cell; another number gives another split and slightly different
scores later on.

In [ ]:
X = perch[["Length", "Height", "Width"]]   ## features: a 2-D table
y = perch["Weight"]                        ## target: weight in grams
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                    random_state=0)
print("train rows:", len(X_train), " test rows:", len(X_test))

### A-3. Fit a linear model
The model is weight ≈ a·length + b·height + c·width + d. Each coefficient is the change in predicted
grams per extra centimetre, with the other two measurements held fixed. The three measurements grow
together, so the model can share the credit between them unevenly: read the coefficients as a fitted
recipe, not as a law of nature.

`fit` picks the coefficients that make the squared errors on the training fish as small as possible
(least squares). Attributes that end in an underscore, such as `coef_` and `intercept_`, are learned from
the data and exist only after `fit`.

In [ ]:
model = LinearRegression().fit(X_train, y_train)
for name, coef in zip(X.columns, model.coef_):
    print(f"{name:9s} {coef:7.1f} g per cm")
print(f"intercept {model.intercept_:7.1f} g")

### A-4. Three metrics
- **MAE** (mean absolute error): the typical size of a miss, here in grams.
- **RMSE** (root mean squared error): like MAE but large misses count more.
- **R²**: the share of the variance in y that the model explains (1.0 is perfect, 0.0 is "predict the mean").

Read the train and test columns side by side. The test column is usually a little worse: the model was
fitted to the training fish and meets the test fish for the first time. A test error far above the
training error would be a warning sign of overfitting, a model that memorised its training rows. RMSE is
never smaller than MAE; the larger the gap between the two, the more of the error comes from a few large
misses.

In [ ]:
def scores(y_true, y_pred):
    return {"MAE": mean_absolute_error(y_true, y_pred),
            "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
            "R2": r2_score(y_true, y_pred)}

table = pd.DataFrame({"train": scores(y_train, model.predict(X_train)),
                      "test": scores(y_test, model.predict(X_test))})
table.round(3)

### A-5. Parity plot
A parity plot puts the true value on the x-axis and the prediction on the y-axis. Perfect predictions
sit on the diagonal; the spread around it is the error you just measured. Points above the line are
over-predictions, points below it under-predictions. Look at the lightest fish:
weight grows roughly with the cube of a fish's size, a straight-line model cannot follow that curve,
and it even predicts negative grams. In class you will compare it with models that can bend.

In [ ]:
y_pred = model.predict(X_test)
fig, ax = plt.subplots(figsize=(4.5, 4.5))
ax.scatter(y_test, y_pred, s=18)
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
ax.plot(lims, lims, "k--", linewidth=1)
ax.set_xlabel("true weight (g)"); ax.set_ylabel("predicted weight (g)")
ax.set_title("Test set parity")
fig.savefig(OUTPUT_DIR / "preclass_parity.png", dpi=150, bbox_inches="tight")
plt.show()

### A-6. Let the line bend: polynomial features
Weight grows roughly with the cube of a fish's size, so we give the linear model curved inputs.
`PolynomialFeatures(degree=2)` adds every square and product of the three measurements (length², length ×
height, ...); degree 3 adds the cubes, and so on. The model is still linear in its coefficients and is fitted
by the same least squares: this is the polynomial regression from the lecture. The powers span many orders of
magnitude (a length⁵ is in the hundreds of millions), so a `StandardScaler` first puts every column on one
scale, and `make_pipeline` chains the three steps; you will meet pipelines again in class.

Read the table from top to bottom. The training RMSE falls with every degree, because more coefficients can
follow the training fish more closely. The test RMSE falls at first: degree 1 **underfits**, it cannot follow
the curve and predicts negative grams. At degree 5 the test RMSE shoots up: 55 coefficients for 44 training
fish, and the model memorises them. That is **overfitting**, and it predicts negative grams again. Picking the
degree from this test column would turn the test set into a tuning set; in class we make such choices with
cross-validation on the training rows.

In [ ]:
rows = {}
for degree in [1, 2, 3, 5]:
    poly = make_pipeline(PolynomialFeatures(degree=degree, include_bias=False),
                         StandardScaler(), LinearRegression())
    poly.fit(X_train, y_train)
    rows[degree] = {"coefficients": poly[0].n_output_features_,
                    "train RMSE": scores(y_train, poly.predict(X_train))["RMSE"],
                    "test RMSE": scores(y_test, poly.predict(X_test))["RMSE"],
                    "lowest test prediction": poly.predict(X_test).min()}
pd.DataFrame(rows).T.rename_axis("degree").round(1).astype({"coefficients": int})

## B. PyTorch: tensors, gradients, a training loop

A neural network is fitted by **gradient descent**: compute how wrong the model is, ask PyTorch for the
gradient of that error with respect to every parameter, and take a small step downhill. Repeat.

### B-1. Tensors
A tensor is PyTorch's array. It converts to and from NumPy, and it remembers which device and dtype it
uses. The training table becomes a 2-D tensor: one row per fish, one column per measurement.
Neural networks normally compute in `float32`; NumPy and pandas default to `float64`, which is why we ask
for `float32` explicitly.

In [ ]:
t = torch.tensor(X_train.to_numpy(), dtype=torch.float32)
print("shape", tuple(t.shape), " dtype", t.dtype)
print("first row", t[0], " back to NumPy:", t.numpy()[0])

### B-2. Automatic gradients
With `requires_grad=True`, PyTorch records every operation on `w`. After `backward()` the derivative of
the loss with respect to `w` is in `w.grad`. Here loss = (3w − 9)², so d(loss)/dw = 2(3w − 9)·3 = −18 at w = 2.
That is the chain rule you would apply by hand; PyTorch does the same bookkeeping for every parameter of a
network, however many there are.

In [ ]:
w = torch.tensor(2.0, requires_grad=True)
loss = (3 * w - 9) ** 2
loss.backward()
print("loss", loss.item(), " gradient", w.grad.item(), " (expected -18.0)")

### B-3. A training loop in ten lines
`nn.Linear(3, 1)` is one neuron with three inputs: the same model as in A-3. Gradient descent works best
when the inputs have similar scales, so we first standardise each measurement with the mean and standard
deviation of the training rows. `SGD` takes the plain gradient step; in class we use Adam, which adapts
the step size for each weight. The same four steps train the multi-layer network in class, where the model
gets bigger and the data come in mini-batches. With standardised inputs the bias ends up at the mean training
weight, which the last line checks.

Two words you will meet again: the **learning rate** `lr` is the size of each step, and an **epoch** is one
pass over the training data (here every epoch uses all training rows at once). `zero_grad()` comes first
because PyTorch adds new gradients to the old ones instead of replacing them. If you try another `lr`: too
small and the loss falls slowly, too large and it can grow instead of falling.

In [ ]:
X_t = torch.tensor(X_train.to_numpy(), dtype=torch.float32)
y_t = torch.tensor(y_train.to_numpy(), dtype=torch.float32).reshape(-1, 1)
mean, std = X_t.mean(dim=0), X_t.std(dim=0)   ## training rows only
X_t = (X_t - mean) / std                       ## same scale for all inputs
torch.manual_seed(0)
net = nn.Linear(3, 1)
optimizer = torch.optim.SGD(net.parameters(), lr=0.1)
loss_fn = nn.MSELoss()
losses = []
for epoch in range(200):
    optimizer.zero_grad()              ## 1. forget the previous gradients
    loss = loss_fn(net(X_t), y_t)      ## 2. how wrong are we?
    loss.backward()                    ## 3. gradient w.r.t. w and b
    optimizer.step()                   ## 4. one step downhill
    losses.append(loss.item())
print(f"bias {net.bias.item():.1f} g",
      f"(mean training weight {y_train.mean():.1f} g)")

### B-4. Watch the loss fall
The loss should drop quickly and flatten out. The assert at the end is the one-line check that training
actually moved in the right direction. The test rows are scaled with the training mean and standard
deviation, and `torch.no_grad()` switches the gradient bookkeeping off while we only predict. The two test
RMSEs come out close but not equal: the loop stops after 200 steps, a little short of the exact
least-squares answer that scikit-learn computes in one go.

**Try it:** once this cell has run, change `lr=0.1` to `lr=0.5` in B-3 and run B-3 and B-4 again. Each step
now overshoots the minimum, so the loss grows instead of falling until it becomes `nan` (not a number), and
the assert stops the cell. With `lr=0.01` the loss falls, only more slowly. Set `lr` back to `0.1` and run B-3
and B-4 once more before you go on.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3.2))
ax.plot(losses)
ax.set_yscale("log"); ax.set_xlabel("epoch"); ax.set_ylabel("MSE loss (g²)")
fig.savefig(OUTPUT_DIR / "preclass_loss.png", dpi=150, bbox_inches="tight")
plt.show()
assert losses[-1] < losses[0], "training did not reduce the loss"

X_te = (torch.tensor(X_test.to_numpy(), dtype=torch.float32) - mean) / std
with torch.no_grad():
    resid = net(X_te).numpy().ravel() - y_test.to_numpy()
print(f"test RMSE  PyTorch {np.sqrt(np.mean(resid ** 2)):.1f} g",
      f"  scikit-learn {table.loc['RMSE', 'test']:.1f} g")

### B-5. Why hidden layers: XOR
XOR answers 1 when exactly one of its two inputs is 1. The lecture showed that a single perceptron cannot
learn it, because no straight line separates the two 1s from the two 0s, and that one hidden layer can. The
cell first plugs in the lecture's hand-set weights, then lets PyTorch train both models from random weights
with the same four-step loop.

The labels are 0 or 1, so the loss is `BCEWithLogitsLoss`: it applies the sigmoid to the output and measures
the cross-entropy, the usual loss for yes/no labels. Each line prints the predicted probability of 1 for the
four inputs. The single neuron can do no better than about 0.5 everywhere; the network with four hidden units
gets all four right.

In [ ]:
X_xor = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
y_xor = torch.tensor([[0.], [1.], [1.], [0.]])   ## 1 when exactly one input is 1

## The lecture's weights: h1 acts as OR, h2 as NAND, and the output as AND of the two.
W1, b1 = torch.tensor([[20., -20.], [20., -20.]]), torch.tensor([-10., 30.])
W2, b2 = torch.tensor([[20.], [20.]]), torch.tensor([-30.])
hidden = torch.sigmoid(X_xor @ W1 + b1)
print("lecture weights ", torch.sigmoid(hidden @ W2 + b2).ravel().round(decimals=2).tolist())

def train_xor(model, steps=2000):
    optimizer = torch.optim.Adam(model.parameters(), lr=0.1)
    loss_fn = nn.BCEWithLogitsLoss()   ## sigmoid + cross-entropy for 0/1 labels
    for step in range(steps):
        optimizer.zero_grad()
        loss = loss_fn(model(X_xor), y_xor)
        loss.backward()
        optimizer.step()
    with torch.no_grad():
        return torch.sigmoid(model(X_xor)).ravel().round(decimals=2).tolist()

torch.manual_seed(0)
print("one neuron      ", train_xor(nn.Linear(2, 1)))
torch.manual_seed(0)
print("one hidden layer", train_xor(nn.Sequential(nn.Linear(2, 4), nn.Sigmoid(),
                                                  nn.Linear(4, 1))))
print("target          ", y_xor.ravel().tolist())

## Data sources
`Data/fish.csv`, copied unchanged from the code repository of Haesun Park's textbook on machine learning and deep learning (Hanbit Media), https://github.com/rickiepark/hg-mldl, MIT License, Copyright (c) 2020 Haesun Park; the licence text and the file's sha256 are in `Data/fish_SOURCE.txt`. The measurements are the fish catch data from Lake Laengelmavesi, Finland (159 fish of seven species), submitted by Juha Puranen to the Journal of Statistics Education data archive. This notebook uses the 56 perch.